# JCI Aso - Plan of Action 2027 Strategic Insights & Survey Analysis
## Educational Data Analytics Notebook

### Overview & Learning Objectives
This notebook provides an in-depth, hands-on exploratory and strategic analysis of the **JCI Aso Plan of Action Survey**. Designed for data analysts, leadership committees, and learning practitioners, this notebook demonstrates:

1. **Survey Data Grain Handling:** How to accurately analyze survey data where multi-select choices have been unnested into an atomic grain without distorting scalar metrics.
2. **Demographic & Engagement Profiling:** Segmenting members by category (Active vs. Alumni), tenure, and activity levels.
3. **Project Portfolio Matrix (2x2 Quadrant):** Evaluating past projects using a dual-metric framework: **Participation Rate** vs. **Average Satisfaction Rating (1-5 Stars)**.
4. **Discontinuation & Opportunity Cost Analysis:** Quantifying member sentiment on underperforming programs to inform resource reallocation.
5. **Strategic Forecasting for 2027:** Ranking demanded innovations and proposed new programs to form the foundational pillars of the 2027 Plan of Action.
6. **Qualitative Sentiment & Thematic Clustering:** Mining unstructured open-ended feedback for actionable institutional improvements.

### Step 1: Environment Setup & Library Imports
We import `pandas` for data manipulation, aggregation, and cross-tabulations.

In [ ]:
import os
import pandas as pd

# Configure display options for rich analytical output
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 1000)

DATA_FILE = "Plan_of_Action_Cleaned_Grain.csv"
print(f"Target data file: {DATA_FILE}")

### Step 2: Ingesting the Cleaned Grain Dataset
Let's load `Plan_of_Action_Cleaned_Grain.csv` and inspect its shape and structure.

**Critical Analytical Concept: Dual-Grain Handling**
- **Choice Grain (`df` - all 111 rows):** Use this for multi-select columns (`Opportunity_Areas`, `Projects_Involved`, `Benefits_Gained`, `New_Ideas`, `New_Projects_Suggested`). Each row represents one selected option.
- **Respondent Grain (`df_respondents` - 22 rows):** Filter where `Choice_Row_Number == 1`. Use this for respondent-level metrics (demographics, ratings, and single text comments) to ensure no double-counting.

In [ ]:
# Load full dataset (Choice Grain)
df = pd.read_csv(DATA_FILE)
print(f"Total Choice-Grain Rows: {len(df)}")
print(f"Total Columns: {len(df.columns)}\n")

# Isolate Respondent-Level Grain (22 distinct respondents)
df_respondents = df[df['Choice_Row_Number'] == 1].copy()
TOTAL_MEMBERS = len(df_respondents)
print(f"Total Unique Respondents: {TOTAL_MEMBERS}")

# Display columns
print("Columns:", list(df.columns))

### Module 1: Member Demographics & Engagement Health
Let's analyze who responded to the survey: membership category, tenure distribution, and self-reported activity level.

In [ ]:
print("=== 1.1 Membership Category ===")
cat_counts = df_respondents['Member_Category'].value_counts()
cat_pcts = (df_respondents['Member_Category'].value_counts(normalize=True) * 100).round(1)
cat_summary = pd.DataFrame({'Count': cat_counts, 'Percentage (%)': cat_pcts})
print(cat_summary)

print("\n=== 1.2 Tenure in JCI Aso ===")
tenure_counts = df_respondents['Tenure'].value_counts()
tenure_pcts = (df_respondents['Tenure'].value_counts(normalize=True) * 100).round(1)
tenure_summary = pd.DataFrame({'Count': tenure_counts, 'Percentage (%)': tenure_pcts})
print(tenure_summary)

print("\n=== 1.3 Self-Reported Activity Level ===")
act_counts = df_respondents['Activity_Level'].value_counts()
act_pcts = (df_respondents['Activity_Level'].value_counts(normalize=True) * 100).round(1)
act_summary = pd.DataFrame({'Count': act_counts, 'Percentage (%)': act_pcts})
print(act_summary)

print("\n=== 1.4 Cross-Tabulation: Tenure vs. Activity Level ===")
ct_tenure_act = pd.crosstab(df_respondents['Tenure'], df_respondents['Activity_Level'], margins=True)
print(ct_tenure_act)

**Demographic Takeaways:**
- **Youth & Energy:** 86.4% of respondents are Active members (ages 18–40), while 13.6% represent experienced Alumni (ages 40+).
- **Strong Foundation:** Over half (54.5%) are in their first 1–3 years of membership, indicating a substantial cohort of newer members eager for development.
- **High Engagement:** 100% of respondents consider themselves either 'Very active' (50%) or 'Somewhat active' (50%). Zero respondents answered 'Not active'.

### Module 2: The Four JCI Areas of Opportunity
JCI provides four core areas of opportunity:
1. Individual Development
2. Business and Entrepreneurship
3. Community Development
4. International Opportunity

Let's quantify what areas members want to benefit more from in the coming year.

In [ ]:
opp_counts = df['Opportunity_Areas'].dropna().value_counts()
opp_pct = ((opp_counts / TOTAL_MEMBERS) * 100).round(1)
opp_df = pd.DataFrame({'Votes': opp_counts, 'Pct_of_Members (%)': opp_pct})
print("=== Member Desired Opportunity Areas ===")
print(opp_df)

print("\n=== Opportunity Preferences by Membership Category ===")
opp_cat_ct = pd.crosstab(df['Opportunity_Areas'], df['Member_Category'], margins=True)
print(opp_cat_ct)

**Opportunity Area Insights:**
- **Individual Development** (63.6%) and **Community Development** (63.6%) tie at the top, followed closely by **Business & Entrepreneurship** (59.1%) and **International Opportunity** (54.5%).
- Demand is remarkably balanced across all four pillars, proving that members seek a holistic JCI experience spanning personal leadership, enterprise building, and global engagement.

### Module 3: Project Portfolio Performance Matrix (The 2x2 Strategic Quadrant)
To evaluate past projects, we analyze two dimensions:
1. **Participation Rate (%):** What percentage of respondents actively participated in the project?
2. **Average Satisfaction Rating (1-5 Stars):** How impactful and well-regarded was the project among members?

Let's compute these metrics across all 8 projects.

In [ ]:
# 1. Compute Participation Counts
proj_part_counts = df['Projects_Involved'].dropna().value_counts()

# 2. Mapping project names to their respective rating columns
PROJECT_RATING_MAP = {
    'Baba and Yara': 'Rate_BabaYara',
    'Secondary School Debate': 'Rate_Debate',
    'Save a Soul': 'Rate_SaveASoul',
    "International Women's Day": 'Rate_InternationalWomenDay',
    'Membership Development Summit (MDS)': 'Rate_MDS',
    'Educate a Child': 'Rate_EducateChild',
    'Quality Leadership Value (QLV)': 'Rate_QualityLeadershipValue',
    'World Down Syndrome Day': 'Rate_WorldDownSyndromeDay'
}

portfolio_data = []
for proj_name, col_name in PROJECT_RATING_MAP.items():
    part_count = proj_part_counts.get(proj_name, 0)
    part_pct = round((part_count / TOTAL_MEMBERS) * 100, 1)
    
    # Compute rating statistics on respondent grain (Row 1)
    ratings = df_respondents[col_name].dropna()
    mean_rating = round(ratings.mean(), 2)
    std_rating = round(ratings.std(), 2)
    pct_5_star = round((len(ratings[ratings == 5]) / len(ratings)) * 100, 1)
    pct_4_or_5 = round((len(ratings[ratings >= 4]) / len(ratings)) * 100, 1)
    
    portfolio_data.append({
        'Project_Name': proj_name,
        'Participants': part_count,
        'Participation_Rate (%)': part_pct,
        'Avg_Rating (★)': mean_rating,
        'Std_Dev': std_rating,
        '5_Star_Pct (%)': pct_5_star,
        '4_or_5_Star_Pct (%)': pct_4_or_5
    })

df_portfolio = pd.DataFrame(portfolio_data)
df_portfolio = df_portfolio.sort_values(by=['Avg_Rating (★)', 'Participation_Rate (%)'], ascending=False).reset_index(drop=True)
print("=== Project Portfolio Performance Summary ===")
print(df_portfolio.to_string(index=False))

In [ ]:
# Classify into Strategic 2x2 Quadrants
AVG_PART = df_portfolio['Participation_Rate (%)'].mean()
AVG_RATE = df_portfolio['Avg_Rating (★)'].mean()
print(f"Benchmark Medians -> Participation: {AVG_PART:.1f}%, Satisfaction: {AVG_RATE:.2f}★\n")

def assign_quadrant(row):
    high_part = row['Participation_Rate (%)'] >= 60.0
    high_rate = row['Avg_Rating (★)'] >= 4.20
    if high_part and high_rate:
        return '1. Star Flagship (High Part, High Sat)'
    elif not high_part and high_rate:
        return '2. High-Impact Niche (Low Part, High Sat)'
    elif high_part and not high_rate:
        return '3. Operational Overhaul (High Part, Low Sat)'
    else:
        return '4. Resource Drain / Underperformer (Low Part, Low Sat)'

df_portfolio['Quadrant'] = df_portfolio.apply(assign_quadrant, axis=1)
print(df_portfolio[['Project_Name', 'Participation_Rate (%)', 'Avg_Rating (★)', 'Quadrant']].to_string(index=False))

### Strategic Takeaways from the Portfolio Matrix:
1. **Star Flagships:**
   - **Baba and Yara** is JCI Aso's crown jewel (81.8% attendance, **4.64★**). 72.7% gave it 5 stars.
   - **Secondary School Debate** holds exceptional academic satisfaction (**4.55★**) with strong school engagement.
   - **Save a Soul** (68.2%, **4.27★**) and **International Women's Day** (68.2%, **4.23★**) are core community anchors.
2. **Operational Overhaul Needed:**
   - **Quality Leadership Value (QLV)** engages 63.6% of members but scores **3.77★** (the second-lowest rating). Members want greater leadership depth and modernized training formats.
3. **Underperformer / Reallocation Candidate:**
   - **World Down Syndrome Day** has the lowest participation (**22.7%**) and lowest rating (**3.23★**). 3 respondents explicitly voted for its discontinuation.

### Module 4: Benefits Gained by Members
What tangible value do members derive from participating in JCI Aso projects?

In [ ]:
benefit_counts = df['Benefits_Gained'].dropna().value_counts()
benefit_pct = ((benefit_counts / TOTAL_MEMBERS) * 100).round(1)
df_benefits = pd.DataFrame({'Count': benefit_counts, 'Pct_of_Members (%)': benefit_pct})
print("=== Tangible Benefits Gained from Projects ===")
print(df_benefits)

**Benefit Realization:**
- **Personal Growth** is #1 by a wide margin (86.4% of all respondents).
- **Fulfilment for Serving a Cause** (68.2%) and **Networking Opportunities** (59.1%) form the emotional and professional value proposition of JCI Aso.

### Module 5: Project Discontinuation & Critical Resource Reallocation
Survey Question 18 asked members: *'Which project/programme do you want us to discontinue in JCI Aso?'*
Survey Question 19 asked for their reasons.

In [ ]:
disc_counts = df['Programs_To_Discontinue'].dropna().value_counts()
print("=== Votes for Project Discontinuation ===")
print(disc_counts)

print("\n=== Discontinuation Qualitative Rationale ===")
disc_reasons = df_respondents[df_respondents['Reason_For_Discontinuation'].notna()][['Response_ID', 'Programs_To_Discontinue', 'Reason_For_Discontinuation']]
for _, r in disc_reasons.iterrows():
    print(f"[{r['Response_ID']}] Program: {r['Programs_To_Discontinue']}")
    print(f"  Reason: {r['Reason_For_Discontinuation']}\n")

**Discontinuation Insight:**
- **World Down Syndrome Day (WDSD)** is the single project targeted for discontinuation.
- **Strategic Rationale (from Respondent R003):** *"Unless we are channeling our efforts towards schools or campuses where people with Down syndrome are abundant and directly interacting with them by providing every support when necessary, it's one project I feel we should use the resources for another activity."*
- **Recommendation:** Discontinue WDSD as a standalone project, or roll its community intent into a consolidated healthcare initiative, freeing committee bandwidth and financial resources for high-priority 2027 programs.

### Module 6: Strategic Horizon 2027 - Innovations & New Programs
What new ideas and programs do members want JCI Aso to pioneer in 2027?

In [ ]:
print("=== 6.1 Desired Innovations for 2027 ===")
innov_counts = df['New_Ideas'].dropna().value_counts()
innov_pct = ((innov_counts / TOTAL_MEMBERS) * 100).round(1)
df_innov = pd.DataFrame({'Votes': innov_counts, 'Pct_of_Members (%)': innov_pct})
print(df_innov)

print("\n=== 6.2 Proposed New Programs for 2027 ===")
prog_counts = df['New_Projects_Suggested'].dropna().value_counts()
prog_pct = ((prog_counts / TOTAL_MEMBERS) * 100).round(1)
df_progs = pd.DataFrame({'Votes': prog_counts, 'Pct_of_Members (%)': prog_pct})
print(df_progs)

**Strategic Demands for 2027:**
1. **The #1 Mandate: Business Clinic (68.2%):**
   - By a wide margin, members voted to introduce a **Business Clinic** to provide entrepreneurship incubation, financial literacy, and SME advisory.
2. **Global Network: International Exchange Programme (54.5%):**
   - Strong appetite for cross-border collaboration and twin-chapter exchanges.
3. **Operational Partnerships & Grants (63.6%):**
   - Moving away from self-funded local activities toward grant-writing and institutional partnerships.

### Module 7: Qualitative Feedback & Voice of the Member
Let's extract and categorize the open-ended text responses regarding areas of improvement and comments for the 2027 Plan of Action.

In [ ]:
print("=== 7.1 Suggested Areas of Improvement (Q22) ===")
improve_resp = df_respondents[df_respondents['Improve_Areas'].notna()][['Response_ID', 'Improve_Areas']]
for _, r in improve_resp.iterrows():
    print(f"[{r['Response_ID']}]: {r['Improve_Areas']}\n")

print("\n=== 7.2 Additional Suggestions & Comments (Q23) ===")
comment_resp = df_respondents[df_respondents['Other_Comments'].notna()][['Response_ID', 'Other_Comments']]
for _, r in comment_resp.iterrows():
    print(f"[{r['Response_ID']}]: {r['Other_Comments']}\n")

### Thematic Analysis of Member Voices:
1. **Onboarding Rigor & Quality Control:**
   - *"More of onboarding, I think this should be stricter to ascertain quality of members and commitment."*
   - Shift from vanity headcount recruitment to value-driven, committed member onboarding.
2. **Local Project Turnout & Engagement:**
   - *"Membership participation in indigenous L.O projects and programmes. Members do not attend LO projects in their numbers..."*
   - Need targeted incentives, committee accountability, and calendar scheduling to boost active turnout.
3. **Alumni–Youth Mentorship Bridge:**
   - *"I also think there should be an avenue creation where younger members can meet the older members, I mean past presidents as well as the senators."*
   - Establish structured quarterly fireside chats and mentorship roundtables.
4. **Brand PR & Social Media Projection:**
   - *"Social media presence and projection for our activities to be more seeing."*
   - External storytelling to position JCI Aso as the premier young leadership organization in Abuja.

### Module 8: 5 Strategic Recommendations for the 2027 Plan of Action

Based on the empirical survey evidence, the 2027 Plan of Action committee should execute across **Five Core Pillars**:

| Pillar | Core Objective | Key Deliverable |
|---|---|---|
| **1. Enterprise & Economy** | Answer the #1 member demand (68.2%) for SME empowerment | Launch the **JCI Aso Business Clinic** (Quarterly business advisory, pitch sessions, tax/legal guidance). |
| **2. Global Alliances & Grants** | Expand external funding & international scope | Form a dedicated **Grant Writing & Strategic Partnerships Taskforce**; initiate an International Twin-Chapter Exchange. |
| **3. Portfolio Rationalization** | Reallocate resources from low-impact initiatives | Phase out **World Down Syndrome Day**; modernize **Quality Leadership Value (QLV)** with interactive workshops. |
| **4. Institutional Mentorship** | Re-engage Alumni and bridge with emerging leaders | Launch the **Senators & Past Presidents Mentorship Roundtable** (quarterly networking & leadership coaching). |
| **5. Onboarding & PR Overhaul** | Enhance member quality and brand visibility | Introduce a 30-day pre-induction commitment milestone; establish an active Digital Media & Communications bureau. |

In [ ]:
# Summary Matrix for Export
print("Strategic Plan of Action 2027 Analysis Complete.")
print(f"Analysis generated from {TOTAL_MEMBERS} validated member responses.")